# 8.6 缺少哪一項資訊時，應先追問？


把工作收窄成「確認日期」。如果請求沒給日期，理想回答是「你希望哪一天？」；如果已給2026-10-03，就回「已確認日期：2026-10-03」。這裡不連接行事曆，也不安排會議，日期是完成這個小任務唯一必要的資訊。

**澄清**是補問會改變下一步的必要條件。有日期還追問日期，會拖延任務；沒有日期卻自選明天，則可能做錯。先把這兩種情況配成資料，才能教「缺才問，夠就做」。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
requests = [
    {"task": "確認日期", "date": None},
    {"task": "確認日期", "date": "2026-10-03"},
]
for request in requests:
    if request["date"] is None:
        target = "你希望哪一天？"
    else:
        target = "已確認日期：" + request["date"]
    print(request, "→", target)

{'task': '確認日期', 'date': None} → 你希望哪一天？
{'task': '確認日期', 'date': '2026-10-03'} → 已確認日期：2026-10-03


`None`表示沒有值，`is None`檢查缺項。`if`與`else`兩個分支按日期是否存在，產生不同目標回答。程式是我們寫好的標註規則，沒有神經模型在判斷模糊語句。

真模型還需要從新的措辭讀出日期，並在回答中保留它。一個保存的小實驗讓我們看見這個區別：新日期題中，缺日期的三題都回「請提供日期」，已給日期的三題卻都確認錯。例如輸入2026-10-01，回答變成2026-10-10。學會固定追問句，並不等於能可靠完成資訊已足夠的工作。

要準備訓練與測試，就同時保留缺資訊和資訊齊全的例子；同一日期的變體一起分組，避免教過日期後又稱它是新材料。換成「確認日期與時間」時，時間也成了必要欄位，原規則就需要擴充。

練習先在紙上列出日期／時間都有、只缺日期、只缺時間、兩者都缺四種情況，再給字典加time欄位。理想回答應只問缺少的項目，兩項齊全則直接確認。

<details>
<summary>補充：既有實驗的條件與完整紀錄</summary>

這種成對日期題也加入了[8.4的同一份模型](https://birdhackor.github.io/tiny-perceptron-vlm/8.4.html)。24個日期各有「日期已給」與「日期缺少」兩題，先把同日期的兩題視為一組，再把不同日期組分給三種用途：19個日期、38題用來更新模型；2個日期、4題用來觀察學習中的表現，叫驗證資料；另外3個日期、6題留到最後檢查，不拿來更新模型。兩題一起分配，避免模型在學習時見過某日期，最後卻又拿同日期的另一題當新題。缺日期題在資料表中仍保留所屬日期的分組標記，但輸入的`date`欄位用`?`代替實際日期，另帶題目編號`id`，不是把分組日期直接告訴模型。

最後檢查的三個日期都未用於訓練，因此是在核對模型能否處理新日期。缺日期的三題都生成`請提供日期。`；但日期已給的三題全確認錯了。例如`task=date;date=2026-10-01;confirm`得到`已確認2026-10-10。`。所以3／6的總分包含兩種很不同的結果：模型學到了固定澄清句，還不能可靠保留新日期。這不是已能安排會議，也沒有測試任意自然語句的缺資訊判斷；它讓我們知道下一步要補哪種能力。

</details>
